# SA Coefficients Viewer

Visualize sensitivity analysis results from a T3 `sa_coefficients.yml` file.

The file stores kinetics SA (dln[species]/dln[k]) and thermo SA (dln[species]/dH[species]) coefficients over time, produced by either the Cantera or RMG adapter.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

from arc.common import read_yaml_file
from t3.common import sa_dict_from_serializable

In [ ]:
# --- Set the path to your sa_coefficients.yml file ---
sa_yaml_path = '../tests/test_simulate/data/sa_coefficients.yml'  # <-- edit this path

raw = read_yaml_file(sa_yaml_path)
sa_dict = sa_dict_from_serializable(raw)
time = sa_dict['time']

metadata = raw.get('metadata', {})
if metadata:
    print('Metadata:')
    for k, v in metadata.items():
        print(f'  {k}: {v}')
    print()

In [ ]:
# --- List available observables ---
kinetics_observables = sorted(sa_dict.get('kinetics', {}).keys())
thermo_observables = sorted(sa_dict.get('thermo', {}).keys())

all_observables = sorted(set(kinetics_observables) | set(thermo_observables))

print('Available observables:')
for i, obs in enumerate(all_observables):
    sections = []
    if obs in kinetics_observables:
        n_rxns = len(sa_dict['kinetics'][obs])
        sections.append(f'kinetics: {n_rxns} reactions')
    if obs in thermo_observables:
        n_spc = len(sa_dict['thermo'][obs])
        sections.append(f'thermo: {n_spc} species')
    print(f'  [{i}] {obs}  ({", ".join(sections)})')

In [ ]:
# --- Select observables to display ---
# Enter indices from the list above, e.g. [0, 2], or 'all'
selected = 'all'  # <-- edit this: list of indices or 'all'

if selected == 'all':
    selected_observables = all_observables
else:
    selected_observables = [all_observables[i] for i in selected]

print(f'Selected observables: {selected_observables}')

## Kinetics SA

Plot dln[species] / dln[k] vs. time for each selected observable.

In [ ]:
# --- Kinetics SA plots ---
for obs in selected_observables:
    if obs not in sa_dict.get('kinetics', {}):
        print(f'No kinetics SA data for {obs}, skipping.')
        continue
    params = sa_dict['kinetics'][obs]
    if not params:
        print(f'Kinetics SA for {obs} is empty, skipping.')
        continue

    fig, ax = plt.subplots(figsize=(10, 5))
    for rxn_index, coeffs in sorted(params.items(), key=lambda x: int(x[0])):
        ax.plot(time, coeffs, label=f'k{rxn_index}')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel(f'dln[{obs}] / dln[k]')
    ax.set_title(f'Kinetics SA — {obs}')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
    fig.tight_layout()
    plt.show()

## Thermo SA

Plot dln[species] / dH[species] vs. time for each selected observable.

In [ ]:
# --- Thermo SA plots ---
for obs in selected_observables:
    if obs not in sa_dict.get('thermo', {}):
        print(f'No thermo SA data for {obs}, skipping.')
        continue
    params = sa_dict['thermo'][obs]
    if not params:
        print(f'Thermo SA for {obs} is empty, skipping.')
        continue

    fig, ax = plt.subplots(figsize=(10, 5))
    for spc_label, coeffs in sorted(params.items()):
        ax.plot(time, coeffs, label=spc_label)
    ax.set_xlabel('Time [s]')
    ax.set_ylabel(f'dln[{obs}] / dH[species]')
    ax.set_title(f'Thermo SA — {obs}')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
    fig.tight_layout()
    plt.show()